In [ ]:
import yaml
import os, pathlib
from itertools import product
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

os.environ['BASE_PERIOD'] = "7"

from python.config import SingleStepConfig, AggregationsConfig
from python.error_metrics import FilteredErrorMetrics, LoadedErrorMetrics, MetricsIndex as MI
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies
from python.aggregation import Aggregation
from python.full_view import CompleteView, Transition, Tendency, Granularity, Dataset, Model, P, EventList, Corr, COMPLETE_DF, CertainEvent, ImpossibleEvent, Event, CombineMethod, CombinePValues
from python.full_view_plots import mosaic, bar, association_matrix, upset, prob_vs_alpha, corr_vs_alpha, combine_significance

STAGNATING, DECREASING, INCREASING = Tendency
INCO_TO_RECO, MIN_TO_SINGLE_ROW, SINGLE_TO_TRIPLE, MIN_TO_TRIPLE_ROWS, SINGLES_TO_MERGED = Transition
BOTTOM, TOP = Granularity
SIM, WEB, M4, M5 = Dataset
AUTOETS, AUTOARIMA, PROPHET = Model
CERTAIN_EVENT, IMPOSSIBLE_EVENT = CertainEvent.CERTAIN_EVENT, ImpossibleEvent.IMPOSSIBLE_EVENT


In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

import warnings
from pandas.errors import PerformanceWarning
warnings.simplefilter("ignore", PerformanceWarning)

In [ ]:
with open('cv/merged_config.yaml', 'r') as merge_f:
    raw = yaml.safe_load(merge_f)
    merged_cfg = AggregationsConfig.model_validate(raw)

with open('cv/single_config.yaml', 'r') as single_f:
    raw = yaml.safe_load(single_f)
    single_cfg = SingleStepConfig.model_validate(raw)

single_sh = SingleStepHierarchies(Aggregation.complete_summing_matrix(n=single_cfg.SUBSET, seed=42))

merge_sh = SampledHierarchies(Aggregation.complete_summing_matrix(n=merged_cfg.SUBSET, seed=42), merged_cfg.DIFF, merged_cfg.SEQUENCES, max_h=merged_cfg.MAX_H)

In [ ]:
if not pathlib.Path('cv/complete_view.csv').exists():
    cv = CompleteView(single_sh, merge_sh,
                lambda d,m: FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{single_cfg.MASTER_FOLDER}/{d}/{m}/sample_hiers_errors.csv"), MI(error_metric='nrmse')),
                lambda d,m: FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{merged_cfg.MASTER_FOLDER}/{d}/{m}/sample_hiers_errors.csv"), MI(error_metric='nrmse')),
                lambda d,m: FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{merged_cfg.MASTER_FOLDER}/{d}/{m}/avg_hiers_erros.csv"), MI(error_metric='nrmse')),
                )
    cv._df.to_csv('cv/complete_view.csv')

In [ ]:
P(DECREASING @ MIN_TO_SINGLE_ROW @ BOTTOM | STAGNATING @ MIN_TO_SINGLE_ROW @ BOTTOM & ~(INCREASING @ MIN_TO_SINGLE_ROW @ BOTTOM))

In [ ]:
P(DECREASING @ MIN_TO_SINGLE_ROW @ TOP | STAGNATING @ MIN_TO_SINGLE_ROW @ TOP & ~(INCREASING @ MIN_TO_SINGLE_ROW @ TOP))

In [ ]:
P((DECREASING @ MIN_TO_SINGLE_ROW @ BOTTOM | STAGNATING @ MIN_TO_SINGLE_ROW @ BOTTOM & ~(INCREASING @ MIN_TO_SINGLE_ROW @ BOTTOM)).given(~SIM))

In [ ]:
P((DECREASING @ MIN_TO_SINGLE_ROW @ BOTTOM | STAGNATING @ MIN_TO_SINGLE_ROW @ BOTTOM & ~(INCREASING @ MIN_TO_SINGLE_ROW @ BOTTOM)) &
  (DECREASING @ MIN_TO_SINGLE_ROW @ TOP | STAGNATING @ MIN_TO_SINGLE_ROW @ TOP & ~(INCREASING @ MIN_TO_SINGLE_ROW @ TOP)))


In [ ]:
G = BOTTOM
T = MIN_TO_SINGLE_ROW

# min-to-single still the best -> above 50 at either gran
# for G, T in product(Granularity, Transition):
mosaic(
    title=f'Tendency distribution per dataset per model at {G} at {T}',
    y_events=[DECREASING @ T @ G,
            STAGNATING @ T @ G,
            INCREASING @ T @ G],
    x_events=EventList(M4, M5, WEB, SIM) * EventList(AUTOETS, AUTOARIMA, PROPHET),
    x_exclusive=True,
    horizontal_line_at=0.5
).show()

In [ ]:
G = BOTTOM
T = MIN_TO_SINGLE_ROW

# Just the ones strictly adhering the theorem

bar(
    y_event=DECREASING @ T @ G | STAGNATING @ T @ G & ~(INCREASING @ T @ G),
    x_events=EventList(M4, M5, WEB, SIM) * EventList(AUTOETS, AUTOARIMA, PROPHET),
)

In [ ]:
G = BOTTOM
T = MIN_TO_SINGLE_ROW

# No clear correlation at any transition

# for G, T in product(Granularity, Transition):
mosaic(
    title=f'Tendency distribution per dataset, given that reconciliation helped or not. At {G} at {T}',
    y_events=[DECREASING @ T @ G,
              STAGNATING @ T @ G,
              INCREASING @ T @ G],
    x_events=EventList(M4, M5, WEB, SIM) * (DECREASING @ INCO_TO_RECO @ G),
    horizontal_line_at=0.5
).show()

In [ ]:
T_X = MIN_TO_SINGLE_ROW
T_Y = SINGLE_TO_TRIPLE
G = BOTTOM
FILTER = CERTAIN_EVENT

# See the correlation matrix rather

# for T_X, T_Y in product(Transition, Transition):
#     if T_X == T_Y: continue
mosaic(
    title=f'Correlation between tendencies at {T_X} and {T_Y}',
    y_events=[DECREASING @ T_Y @ G,
              STAGNATING @ T_Y @ G,
              INCREASING @ T_Y @ G] ,
    x_events=EventList(DECREASING @ T_X @ G,
                       STAGNATING @ T_X @ G,
                       INCREASING @ T_X @ G) & FILTER,
    ignore_none=True,
).show()

In [ ]:
for G in Granularity:
    association_matrix(events=EventList(DECREASING @ INCO_TO_RECO @ G,
                                        DECREASING @ MIN_TO_SINGLE_ROW @ G,
                                        DECREASING @ MIN_TO_TRIPLE_ROWS @ G,
                                        DECREASING @ SINGLE_TO_TRIPLE @ G,
                                        DECREASING @ SINGLES_TO_MERGED @ G),
                                        method='phi', title=G).show()

In [ ]:
G = BOTTOM

upset(EventList(DECREASING @ INCO_TO_RECO @ G,
                DECREASING @ MIN_TO_SINGLE_ROW @ G,
                DECREASING @ MIN_TO_TRIPLE_ROWS @ G,
                DECREASING @ SINGLE_TO_TRIPLE @ G,
                DECREASING @ SINGLES_TO_MERGED @ G))

In [ ]:
T = MIN_TO_SINGLE_ROW
prob_vs_alpha([DECREASING @ T @ BOTTOM | STAGNATING @ T @ BOTTOM,
               DECREASING @ T @ BOTTOM,
               STAGNATING @ T @ BOTTOM]).show()

In [ ]:
corr_vs_alpha((DECREASING @ MIN_TO_SINGLE_ROW @ BOTTOM, DECREASING @ SINGLE_TO_TRIPLE @ BOTTOM))

In [ ]:
combine_significance(
    EventList(*[
        DECREASING @ tr @ g
        for g in Granularity
        for tr in (MIN_TO_SINGLE_ROW, MIN_TO_TRIPLE_ROWS, SINGLE_TO_TRIPLE, SINGLES_TO_MERGED)
    ]).given(EventList(M4, M5, WEB, SIM, CERTAIN_EVENT)),
    EventList(*[
        INCREASING @ tr @ g
        for g in Granularity
        for tr in (MIN_TO_SINGLE_ROW, MIN_TO_TRIPLE_ROWS, SINGLE_TO_TRIPLE, SINGLES_TO_MERGED)
    ]).given(EventList(M4, M5, WEB, SIM, CERTAIN_EVENT)),
    methods=[CombineMethod.FISHER, CombineMethod.STOUFFER, CombineMethod.MUDHOLKAR_GEORGE]
)